In [ ]:
# ============================================
# Pinecone Serverless Reranking in Action
# ============================================

# ============================================================
# Part 1: Load Documents and Execute Reranking Model
# ============================================================

# 1. Install Pinecone libraries (à exécuter dans une cellule séparée si besoin)
# -------------------------------------------------------------------------
# !pip install -U pinecone==6.0.1 pinecone-notebooks
# (recommande aussi : !pip install pandas torch transformers)

# 2. Authenticate with Pinecone
# -------------------------------------------------------------------------
import os

if not os.environ.get("PINECONE_API_KEY"):
    from pinecone_notebooks.colab import Authenticate
    Authenticate()

# 3. Instantiate the Pinecone client
# -------------------------------------------------------------------------
from pinecone import Pinecone

api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=api_key)

# 4. Define your query and documents
# -------------------------------------------------------------------------
query = "Tell me about Apple's products"

documents = [
    "Apple is a sweet, crunchy fruit that comes in many varieties such as Granny Smith and Fuji. It is often eaten raw or baked in pies.",
    "Apple Inc. is a technology company that designs and sells products like the iPhone, iPad, MacBook, and Apple Watch.",
    "Many people enjoy eating an apple as a healthy snack during the day because it is rich in fiber and vitamins.",
    "The company Apple is known for its innovative products, including smartphones, laptops, tablets, and various subscription services.",
    "Some nutritionists recommend eating at least one apple per day as part of a balanced diet."
]

# 5. Call the reranker
# -------------------------------------------------------------------------
from pinecone import RerankModel  # non utilisé directement mais conservé pour suivre l énoncé

reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
    top_n=3,  # par exemple : trois meilleurs résultats
    return_documents=True
)

# 6. Inspect reranked results
# -------------------------------------------------------------------------
def show_reranked_results(query, matches):
    print(f"Query: {query}")
    print("\nReranked documents:")
    for i, m in enumerate(matches):
        # m.score et m.document.text selon la structure RerankResult
        print(f"{i + 1}. Score: {m.score:.4f}")
        print(f"   Document id: {m.document.id}")
        print(f"   Text: {m.document.text}")
        print()

show_reranked_results(query, reranked.data)


# ============================================================
# Part 2: Setup a Serverless Index for Medical Notes
# ============================================================

# 1. Install data and model libraries (à exécuter dans une cellule séparée si besoin)
# -------------------------------------------------------------------------
# !pip install pandas torch transformers

# 2. Import modules and define environment settings
# -------------------------------------------------------------------------
import time
import pandas as pd
from pinecone import ServerlessSpec
from transformers import AutoTokenizer, AutoModel
import torch
import requests
import tempfile

# Get cloud and region settings (defaults)
cloud = os.getenv("PINECONE_CLOUD", "aws")         # par exemple "aws"
region = os.getenv("PINECONE_REGION", "us-east-1") # par exemple "us-east-1"

# Define serverless specifications
spec = ServerlessSpec(cloud=cloud, region=region)

# Define index name
index_name = "medical-notes-index"  # nom libre de ton choix

# 3. Create or recreate the index
# -------------------------------------------------------------------------
# Clean up any existing index with the same name
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)

# Create a new index
pc.create_index(
    name=index_name,
    dimension=384,        # correspond à all-MiniLM-L6-v2
    metric="cosine",      # mesure recommandée pour des embeddings texte
    spec=spec
)


# ============================================================
# Part 3: Load the Sample Data
# ============================================================

# 1. Download and read JSONL
# -------------------------------------------------------------------------
with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = os.path.join(tmpdirname, "sample_notes_data.jsonl")

    # Download the file from GitHub
    url = "https://raw.githubusercontent.com/pinecone-io/examples/refs/heads/master/docs/data/sample_notes_data.jsonl"
    response = requests.get(url)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

    df = pd.read_json(file_path, orient="records", lines=True)

# 2. Preview the DataFrame
# -------------------------------------------------------------------------
print("Data shape:", df.shape)  # nombre de lignes et de colonnes
display(df.head())


# ============================================================
# Part 4: Upsert Data into the Index
# ============================================================

# 1. Instantiate index client and upsert
# -------------------------------------------------------------------------
index = pc.Index(name=index_name)

# Upsert data into index from DataFrame
index.upsert_from_dataframe(df=df)

# 2. Wait for availability
# -------------------------------------------------------------------------
def is_fresh(index):
    stats = index.describe_index_stats()
    vector_count = stats.total_vector_count
    print("Vector count: ", vector_count)
    # attendre au moins un vecteur inséré
    return vector_count > 0

while not is_fresh(index):
    time.sleep(5)

print("Index ready!")
print(index.describe_index_stats())


# ============================================================
# Part 5: Query and Embedding Function
# ============================================================

# 1. Define your embedding function
# -------------------------------------------------------------------------
def get_embedding(input_question):
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)

    encoded_input = tokenizer(
        input_question,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )
    with torch.no_grad():
        model_output = model(**encoded_input)
        # last_hidden_state: (batch_size, seq_len, hidden_size)
        # on moyenne sur la dimension seq_len pour obtenir un vecteur
        embedding = model_output.last_hidden_state[0].mean(dim=0)
    return embedding


# 2. Run a semantic search query
# -------------------------------------------------------------------------
# Build a query to search
question = "What is the recommended treatment for a patient with chest pain and shortness of breath?"
query_vector = get_embedding(question).tolist()

# Get results
results = index.query(
    vector=query_vector,    # un seul vecteur
    top_k=5,                # nombre de résultats
    include_metadata=True
)

# Sort results by score in descending order
sorted_matches = sorted(results["matches"], key=lambda x: x["score"], reverse=True)


# ============================================================
# Part 6: Display and Rerank Clinical Notes
# ============================================================

# 1. Display initial search results
# -------------------------------------------------------------------------
def show_results(question, matches):
    print(f"Question: '{question}'")
    print("\nResults:")
    for i, match in enumerate(matches):
        print(f"{str(i + 1).rjust(4)}. ID: {match['id']}")
        print(f" Score: {match['score']}")
        print(f" Metadata: {match['metadata']}")
        print("")

show_results(question, sorted_matches)


# 2. Prepare documents for reranking
# -------------------------------------------------------------------------
# Create documents with concatenated metadata field as "reranking_field"
transformed_documents = [
    {
        "id": match["id"],
        "reranking_field": "; ".join(
            [f"{key}: {value}" for key, value in match["metadata"].items()]
        ),
    }
    for match in results["matches"]
]


# 3. Execute serverless reranking
# -------------------------------------------------------------------------
# Define a more specific query for reranking
refined_query = "Patient with severe chest pain, possible myocardial infarction, needs immediate treatment plan."

# Perform reranking based on the query and specified field
reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],
    top_n=3,  # par exemple trois meilleurs résultats
    return_documents=True,
)


# 4. Show reranked results
# -------------------------------------------------------------------------
def show_reranked_results_clinical(question, matches):
    print(f"Question: '{question}'")
    print("\nReranked Results:")
    for i, match in enumerate(matches):
        print(f"{str(i + 1).rjust(4)}. ID: {match.document.id}")
        print(f" Score: {match.score}")
        print(f" Reranking Field: {match.document.reranking_field}")
        print("")

show_reranked_results_clinical(refined_query, reranked_results.data)


# 5. Clean up (optional)
# -------------------------------------------------------------------------
# À exécuter uniquement lorsque tu as terminé tes essais
# pc.delete_index(name=index_name)
